# Model Router Toolkit — Quickstart

Route every LLM call to the **cheapest model that can answer it correctly**.

This notebook walks through the inference path: load a pre-trained checkpoint, route queries, inspect results, and serve.
No training, no API keys (until serving), no GPU required.

**Prerequisites:** Python 3.10+, ~1.6 GB disk for the encoder model (downloaded from HuggingFace on first use).

> **Git LFS required:** Checkpoints are stored with Git LFS. If you haven't already, run `git lfs install && git lfs pull` from the repo root. Without this, checkpoint files will be tiny pointer files and loading will fail.

---
## 1. Install

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Working directory: {os.getcwd()}")

In [ ]:
%pip install -q -e '.[prefill,litellm]'

---
## 2. Route a Query (Python)

Load the config and checkpoint, then route a question. No API key, no server, no network — runs entirely locally.

In [ ]:
from model_router_toolkit.config import load_config, build_router_from_config

config = load_config("configs/v1-9models-qwen08b.yaml")
router = build_router_from_config(config)
print(f"Loaded router with {len(config.model_names)} models")
print(f"Models: {config.model_names}")

In [ ]:
result = router.route("What is the capital of France?", tolerance=0.20)

print(f"Selected model: {result.selected_model}")
print(f"Selected confidence: {result.selected_confidence:.3f}")
print(f"Tolerance: {result.metadata['tolerance']}")
print(f"Threshold: {result.metadata['threshold']:.3f}")
print()
print("Per-model confidences:")
for name, conf in zip(result.model_names, result.confidences):
    marker = " <-- selected" if name == result.selected_model else ""
    print(f"  {name:30s}  P(correct)={conf:.3f}{marker}")

---
## 3. Try Different Tolerances

The `tolerance` parameter controls the accuracy-cost tradeoff:
- `0.0` — always pick the highest-confidence model (most expensive)
- `0.20` (default) — allow up to 20pp below the best for a cheaper model
- `1.0` — always pick the cheapest model

In [ ]:
question = "Prove that the square root of 2 is irrational"

print(f"Question: {question}\n")
print(f"{'Tolerance':>10s}  {'Selected Model':30s}  {'Confidence':>10s}  {'Cost ($/M in)':>13s}")
print(f"{'─' * 10}  {'─' * 30}  {'─' * 10}  {'─' * 13}")

for tol in [0.0, 0.10, 0.20, 0.50, 1.0]:
    r = router.route(question, tolerance=tol)
    cost = r.selected_cost.cost_per_m_input_tokens
    print(f"{tol:10.2f}  {r.selected_model:30s}  {r.selected_confidence:10.3f}  ${cost:12.3f}")

---
## 4. Route Multiple Questions

Route a batch of questions varying in difficulty and see how the router distributes traffic.

In [ ]:
from collections import Counter

questions = [
    "What is 2 + 2?",
    "What is the capital of France?",
    "Explain the difference between TCP and UDP",
    "Write a Python function to compute Fibonacci numbers using memoization",
    "Prove that the square root of 2 is irrational",
    "Explain the P vs NP problem and its implications",
    "Derive the Euler-Lagrange equation from the principle of least action",
]

print(f"{'Question':55s}  {'Selected Model':30s}  {'Conf':>5s}")
print(f"{'─' * 55}  {'─' * 30}  {'─' * 5}")

selections = []
for q in questions:
    r = router.route(q, tolerance=0.20)
    selections.append(r.selected_model)
    print(f"{q[:55]:55s}  {r.selected_model:30s}  {r.selected_confidence:.3f}")

print("\nRouting distribution:")
for model, count in Counter(selections).most_common():
    print(f"  {model}: {count}/{len(questions)} ({count/len(questions)*100:.0f}%)")

---
## 5. Model Subset

Restrict routing to a subset of models using the `models` parameter.

In [ ]:
subset = ["nemotron-3-nano-reasoning", "qwen-3-5-122b", "claude-opus-4-6-high"]

r_full = router.route("Explain quantum entanglement", tolerance=0.20)
r_sub = router.route("Explain quantum entanglement", tolerance=0.20, models=subset)

print(f"Full pool ({len(config.model_names)} models): {r_full.selected_model}")
print(f"Subset    ({len(subset)} models): {r_sub.selected_model}")
print(f"\nSubset confidences:")
for name, conf in zip(r_sub.model_names, r_sub.confidences):
    if name in subset:
        marker = " <-- selected" if name == r_sub.selected_model else ""
        print(f"  {name:30s}  {conf:.3f}{marker}")

---
## 6. Serve the Router

Start the full server — it routes **and** calls the selected model via OpenRouter. The server exposes:
- OpenAI-compatible API at `/v1/chat/completions`
- Playground UI at `http://localhost:8000/`

You'll need an [OpenRouter API key](https://openrouter.ai/keys).

In [ ]:
import os
if not os.environ.get("OPENROUTER_API_KEY"):
    api_key = input("Enter your OpenRouter API key (https://openrouter.ai/keys): ")
    os.environ["OPENROUTER_API_KEY"] = api_key
print(f"OPENROUTER_API_KEY set ({len(os.environ['OPENROUTER_API_KEY'])} chars)")

In [ ]:
!model-router serve \
    --config configs/v1-9models-qwen08b.yaml \
    --port 8000 &

import time, requests
print("Waiting for server to load encoder...")
for attempt in range(30):
    time.sleep(5)
    try:
        resp = requests.get("http://localhost:8000/health", timeout=3)
        if resp.ok:
            print(f"Server ready after {(attempt + 1) * 5}s")
            print(f"  API:        http://localhost:8000/v1/chat/completions")
            print(f"  Playground: http://localhost:8000/")
            break
    except requests.ConnectionError:
        pass
else:
    print("Server did not start within 150s.")

### Call the API

The server is OpenAI-compatible. Send `"model": "routed"` and the router picks the best model for each query.

In [ ]:
!curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "routed", "messages": [{"role": "user", "content": "What is the capital of France?"}]}' \
  | python -m json.tool

---
## Cleanup

In [ ]:
router.unload()
!kill $(lsof -ti :8000) 2>/dev/null && echo "Server stopped." || echo "No server running on port 8000."

---
## What's Next

| Topic | Doc |
|-------|-----|
| Full config reference | [Configuration](../docs/guide-configuration.md) |
| Training & evaluation | [Training Guide](../docs/guide-training-and-evaluation.md) |
| Adapters (LiteLLM, HTTP sidecar) | [Adapters Guide](../docs/guide-adapters-and-plugins.md) |
| Serving & deployment | [Deployment Guide](../docs/guide-serving-and-deployment.md) |
| Architecture and inference flow | [How It Works](../docs/guide-how-it-works.md) |